# Get activations (without debiasing)

Extracts CLIP ViT-Large CLS-token activations from all 24 layers for every image in `data/images/`,
and the final CLIP image embedding (output of `visual_projection`).

**Run once** - output is concept-agnostic and shared across all downstream notebooks.

| output | schema |
|---|---|
| `data/activations/raw/train/layer_XX.parquet` | `filename, 0..1023` (float16) |
| `data/activations/raw/test/layer_XX.parquet`  | `filename, 0..1023` (float16) |
| `data/activations/raw/train/model_output.parquet` | `filename, 0..767` (float16) |
| `data/activations/raw/test/model_output.parquet`  | `filename, 0..767` (float16) |

`model_output` = raw 768-dim CLIP image embedding (after `post_layernorm` + `visual_projection`, before L2 norm).

In [1]:
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
from PIL import Image
from dotenv import load_dotenv
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

In [2]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_ID            = 'openai/clip-vit-large-patch14'
GPU_BATCH_SIZE      = 64
NUM_WORKERS         = 4
PARQUET_COMPRESSION = 'snappy'
WRITE_THREADS       = 4

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent

METADATA_PATH   = ROOT / 'data' / 'metadata.csv'
IMAGES_DIR      = ROOT / 'data' / 'images'
ACTIVATIONS_DIR = ROOT / 'data' / 'activations' / 'raw'

assert METADATA_PATH.exists(), f'Run download_data.ipynb first. Missing: {METADATA_PATH}'
assert IMAGES_DIR.exists(),    f'Run download_data.ipynb first. Missing: {IMAGES_DIR}'

for split in ('train', 'test'):
    (ACTIVATIONS_DIR / split).mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if device == 'cuda' else torch.float32
print(f'Device: {device} | dtype: {dtype}')
print(f'Output: {ACTIVATIONS_DIR}')

Device: cuda | dtype: torch.bfloat16
Output: /workspace/WB2/data/activations/raw


In [4]:
processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
).to(device).eval()
print(f'Model loaded: {MODEL_ID}')


class CelebADataset(Dataset):
    def __init__(self, df, images_dir, processor):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.processor  = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.iloc[idx]['filename']
        img = Image.open(self.images_dir / filename).convert('RGB')
        px  = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        return px, filename


def _write_layer_parquet(l_idx, acts_list, filenames, out_path):
    all_acts = np.concatenate(acts_list, axis=0).astype('float16')
    df_layer = pd.DataFrame(all_acts)
    df_layer.columns = df_layer.columns.astype(str)
    df_layer.insert(0, 'filename', filenames)
    df_layer.to_parquet(out_path, compression=PARQUET_COMPRESSION, index=False)
    return l_idx, os.path.getsize(out_path) / 1024**2

config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Model loaded: openai/clip-vit-large-patch14


In [5]:
def run_extraction(split_name):
    out_dir        = ACTIVATIONS_DIR / split_name
    out_model_path = out_dir / 'model_output.parquet'

    df_meta  = pd.read_csv(METADATA_PATH)
    df_split = df_meta[df_meta['split'] == split_name].reset_index(drop=True)

    # ── Discover number of layers (lazy: load model only once globally) ─────
    n_layers = len(model.vision_model.encoder.layers)

    # ── Resume logic: skip only if ALL expected per-layer parquets exist ───
    expected_layer_files = [out_dir / f'layer_{i:02d}.parquet' for i in range(n_layers)]
    if out_model_path.exists() and all(p.exists() for p in expected_layer_files):
        try:
            cols = pd.read_parquet(expected_layer_files[0]).columns.tolist()
            if 'filename' in cols:
                print(f'[{split_name}] All {n_layers} layers + model_output present — skipping.')
                return
        except Exception:
            pass

    dataset = CelebADataset(df_split, IMAGES_DIR, processor)
    loader  = DataLoader(
        dataset,
        batch_size=GPU_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device == 'cuda'),
        persistent_workers=(NUM_WORKERS > 0),
    )

    acts_buf      = {i: [] for i in range(n_layers)}
    outputs_buf   = []
    filenames_buf = []

    def get_hook_fn(layer_idx):
        def hook_fn(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            acts = hidden[:, 0, :].detach().to(torch.float32).cpu().numpy()
            acts_buf[layer_idx].append(acts)
        return hook_fn

    def output_hook_fn(module, input, output):
        outputs_buf.append(output.detach().to(torch.float32).cpu().numpy())

    handles = [
        model.vision_model.encoder.layers[i].register_forward_hook(get_hook_fn(i))
        for i in range(n_layers)
    ]
    handles.append(model.visual_projection.register_forward_hook(output_hook_fn))

    try:
        with torch.no_grad():
            for pixels, fnames in tqdm(loader, desc=f'Extracting {split_name}'):
                model.get_image_features(pixel_values=pixels.to(device, dtype=dtype))
                filenames_buf.extend(fnames)
    finally:
        for h in handles:
            h.remove()

    with ThreadPoolExecutor(max_workers=WRITE_THREADS) as ex:
        futures = []
        for l_idx in range(n_layers):
            out_path = out_dir / f'layer_{l_idx:02d}.parquet'
            acts = acts_buf.pop(l_idx)
            futures.append(ex.submit(_write_layer_parquet, l_idx, acts, filenames_buf, out_path))

        pbar = tqdm(as_completed(futures), total=n_layers, desc=f'Writing {split_name}')
        for fut in pbar:
            l_idx, size_mb = fut.result()
            pbar.set_postfix_str(f'layer_{l_idx:02d} = {size_mb:.1f} MB')

    # Save final 768-dim CLIP image embedding (post_layernorm + visual_projection, before L2 norm)
    _write_layer_parquet(-1, outputs_buf, filenames_buf, out_model_path)
    print(f'[{split_name}] model_output.parquet saved ({os.path.getsize(out_model_path)/1024**2:.1f} MB)')

In [6]:
for split in ('train', 'test'):
    run_extraction(split)

Extracting train:   0%|          | 0/73 [00:00<?, ?it/s]

Extracting train:   1%|▏         | 1/73 [00:02<02:53,  2.41s/it]

Extracting train:   3%|▎         | 2/73 [00:02<01:18,  1.10s/it]

Extracting train:   4%|▍         | 3/73 [00:02<00:48,  1.45it/s]

Extracting train:   5%|▌         | 4/73 [00:02<00:34,  2.02it/s]

...

Writing test:  92%|█████████▏| 22/24 [00:02<00:00,  7.61it/s, layer_21 = 2.2 MB]

Writing test:  92%|█████████▏| 22/24 [00:03<00:00,  7.61it/s, layer_22 = 2.2 MB]

Writing test: 100%|██████████| 24/24 [00:03<00:00,  8.59it/s, layer_22 = 2.2 MB]

Writing test: 100%|██████████| 24/24 [00:03<00:00,  7.98it/s, layer_22 = 2.2 MB]

[test] model_output.parquet saved (1.7 MB)
